# Conditional VAE with ChestXray14

This is one notebook in a series of three about VAEs:

1. [Synthetic image VAE](vae_cxr_synthetic.ipynb)
2. [ChestXray14 image VAE (this notebook)](vae_cxr_chestxray14.ipynb)
3. [EHR code-sequence VAE](../mortality_prediction/vae_ehr_sequences.ipynb)

This notebook applies PyHealth's image VAE to chest X-rays from the NIH ChestXray14 dataset. It uses the `infiltration` finding as a binary condition, trains the VAE to reconstruct images, and samples images conditioned on the two labels. A VAE reconstructs images; it is not trained here to predict the finding.

The first run downloads the first ChestXray14 image archive through PyHealth (4,999 images) and its label file. This is enough to try the workflow locally; the full dataset contains 112,120 images and requires substantially more storage. If the expected CSV and `images/` directory are already present, the setup cell reuses them without downloading again. `dev=True` limits processing to 1,000 patients after the data are available.

As another option, PyHealth supports the COVID-19 Radiography Database. This notebook does not download it. Download and extract the dataset from [Kaggle](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database), then set `root` in `COVID19CXRDataset(root=...)` to the extracted `COVID-19_Radiography_Dataset` directory. That directory should contain the class folders and metadata spreadsheets (for example, `COVID/`, `Normal/`, `Lung_Opacity/`, `Viral Pneumonia/`, and their metadata files). You need a Kaggle account and must accept the dataset access terms on Kaggle first.

## 1. Configure data and load ChestXray14

We resize images to 64×64 grayscale to keep this example lighter. The setup cell asks PyHealth to download only the first archive; change `PARTIAL_DOWNLOAD` to `False` only if you intend to download the full dataset.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from pyhealth.datasets import (
    ChestXray14Dataset,
    get_dataloader,
    split_by_patient,
)
from pyhealth.models import VAE
from pyhealth.processors import ImageProcessor
from pyhealth.tasks import ChestXray14BinaryClassification
from pyhealth.trainer import Trainer

DATA_ROOT = Path(
    os.environ.get("PYHEALTH_CHESTXRAY14_ROOT", "./data/chestxray14")
)
PARTIAL_DOWNLOAD = True  # First archive: 4,999 images. False downloads all 112,120.
DEV_MODE = True           # Process at most 1,000 patients.
DATA_AVAILABLE = (
    (DATA_ROOT / "Data_Entry_2017_v2020.csv").is_file()
    and (DATA_ROOT / "images").is_dir()
)
DOWNLOAD_DATA = not DATA_AVAILABLE  # Download once; reuse the local copy on reruns.

if DOWNLOAD_DATA:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
elif not DATA_AVAILABLE:
    raise FileNotFoundError(
        f"ChestXray14 files were not found under {DATA_ROOT}. Set "
        "PYHEALTH_CHESTXRAY14_ROOT to your local dataset directory, or set "
        "DOWNLOAD_DATA=True and choose whether to download the partial archive."
    )

base_dataset = ChestXray14Dataset(
    root=str(DATA_ROOT),
    download=DOWNLOAD_DATA,
    partial=PARTIAL_DOWNLOAD,
    dev=DEV_MODE,
)
print(f"ChestXray14 root: {DATA_ROOT}")

## 2. Create a patient-disjoint task split

The binary task marks each image as positive or negative for `infiltration`. The task below adds the patient ID to each sample so PyHealth can build a patient-level split; this keeps images from one person together in one split.

In [ ]:
class PatientAwareChestXrayTask(ChestXray14BinaryClassification):
    task_name = "ChestXray14BinaryClassificationWithPatientId"

    def __call__(self, patient):
        patient_samples = super().__call__(patient)
        for sample in patient_samples:
            sample["patient_id"] = patient.patient_id
        return patient_samples


task = PatientAwareChestXrayTask(disease="infiltration")
samples = base_dataset.set_task(
    task,
    input_processors={
        "image": ImageProcessor(image_size=64, mode="L"),
    },
)

train_samples, val_samples, test_samples = split_by_patient(
    samples, ratios=[0.8, 0.1, 0.1], seed=7
)
print(
    f"Samples - train: {len(train_samples):,}, "
    f"validation: {len(val_samples):,}, test: {len(test_samples):,}"
)
print("Finding label vocabulary:", samples.output_processors["label"].label_vocab)

## 3. Train a conditional image VAE

The `label` condition tells the decoder which binary group to sample from. The training objective is reconstruction plus the VAE's latent regularization term. It does not calculate classification accuracy or establish that generated X-rays are medically realistic.

In [ ]:
BATCH_SIZE = 8
EPOCHS = 3
STEPS_PER_EPOCH = 20

train_loader = get_dataloader(
    train_samples, batch_size=BATCH_SIZE, shuffle=True
)
val_loader = get_dataloader(val_samples, batch_size=BATCH_SIZE)
test_loader = get_dataloader(test_samples, batch_size=BATCH_SIZE)

model = VAE(
    dataset=train_samples,
    feature_key="image",
    condition_key="label",
    hidden_dim=32,
    channel_dim=16,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
trainer = Trainer(
    model=model,
    device=device,
    metrics=[],
    enable_logging=False,
)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=EPOCHS,
    steps_per_epoch=min(STEPS_PER_EPOCH, len(train_loader)),
    optimizer_params={"lr": 1e-3},
)
test_scores = trainer.evaluate(test_loader)
print(f"Test reconstruction objective: {test_scores['loss']:.2f}")

## 4. Inspect reconstructions

Compare original test images with the VAE reconstructions. This is a visual check of the learned reconstruction behavior, not a diagnostic-quality assessment.

In [ ]:
test_batch = next(iter(test_loader))
model.eval()
with torch.no_grad():
    result = model(**test_batch)

label_names = {0: "No infiltration", 1: "Infiltration"}
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index in range(4):
    label = int(test_batch["label"][index].item())
    axes[0, index].imshow(test_batch["image"][index, 0], cmap="gray")
    axes[0, index].set_title(f"Original: {label_names[label]}")
    axes[0, index].axis("off")
    axes[1, index].imshow(
        result["reconstruction"][index, 0].cpu(), cmap="gray", vmin=0, vmax=1
    )
    axes[1, index].set_title("Reconstruction")
    axes[1, index].axis("off")
plt.tight_layout()
plt.show()

## 5. Generate images for both conditions

These samples are drawn from the trained decoder conditioned on the task's two label indices. Treat them as a model-behavior illustration; they are not validated synthetic clinical data.

In [ ]:
model.eval()
conditions = torch.tensor([0, 1], device=model.device)
with torch.no_grad():
    generated = model.generate(conditions=conditions)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for axis, image, label in zip(axes, generated, (0, 1)):
    axis.imshow(image[0].cpu(), cmap="gray", vmin=0, vmax=1)
    axis.set_title(label_names[label])
    axis.axis("off")
plt.tight_layout()
plt.show()

## What this example does and does not show

This notebook demonstrates that PyHealth's VAE can be wired to its ChestXray14 image pipeline and trained conditionally. Its short run and small development subset are for trying the workflow locally. To assess image fidelity or any downstream usefulness, use a suitable patient-level study design, train longer as appropriate, and evaluate the results with domain experts and held-out data.